<a href="https://colab.research.google.com/github/Le2se0hy/XAI_An/blob/main/XGBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor
from google.colab import drive

# ============================================================
# 0. XGBoost 설치
# Colab에서 설치되지 않은 경우에만 주석 해제
# ============================================================
# !pip install xgboost openpyxl -q


# ============================================================
# 1. 데이터 로드
# ============================================================
drive.mount("/content/drive")

file_path = "/content/drive/MyDrive/Seoul_Aprtment_FINAL.xlsx"
df_raw = pd.read_excel(file_path)


# ============================================================
# 2. 전처리 함수
# ============================================================
def prepare_seoul_df_final(df_input):
    df = df_input.copy()

    # --------------------------------------------------------
    # Size
    # --------------------------------------------------------
    if "Size_m2" in df.columns:
        df["Size_Level"] = pd.to_numeric(
            df["Size_m2"],
            errors="coerce"
        )

    # --------------------------------------------------------
    # Population Density
    # --------------------------------------------------------
    if "Pop. Density" in df.columns:
        df["Pop_Density_Level"] = pd.to_numeric(
            df["Pop. Density"],
            errors="coerce"
        )

    # --------------------------------------------------------
    # Units
    # --------------------------------------------------------
    if "num_of_people" in df.columns:
        df["Units_Level"] = pd.to_numeric(
            df["num_of_people"],
            errors="coerce"
        )

    # --------------------------------------------------------
    # 인구 관련 비율 변수
    # Median age, Young Population, Old Population은
    # 이미 0.13 형태로 저장되어 있으므로 그대로 사용
    # --------------------------------------------------------
    ratio_map = {
        "Median age": "Medium_age_ratio",
        "Young Population": "Young_pop_ratio",
        "Old Population": "Old_pop_ratio"
    }

    for original, new in ratio_map.items():
        if original in df.columns:
            df[new] = pd.to_numeric(
                df[original],
                errors="coerce"
            )

    # --------------------------------------------------------
    # 성비
    # 0~1 형태이면 그대로 사용하고,
    # 0~100 형태이면 100으로 나누어 사용
    # --------------------------------------------------------
    if "Sex_ratio" in df.columns:
        sex_ratio = pd.to_numeric(
            df["Sex_ratio"],
            errors="coerce"
        )

        valid_sex_ratio = sex_ratio.dropna()

        if (
            len(valid_sex_ratio) > 0
            and valid_sex_ratio.abs().max() > 2.0
        ):
            df["Sex_ratio_ratio"] = sex_ratio / 100.0
        else:
            df["Sex_ratio_ratio"] = sex_ratio

    # --------------------------------------------------------
    # CBD 거리: m → km
    # --------------------------------------------------------
    if "Dist_CBD" in df.columns:
        df["Dist_CBD_km"] = (
            pd.to_numeric(
                df["Dist_CBD"],
                errors="coerce"
            )
            / 1000.0
        )

    # --------------------------------------------------------
    # 계절 더미변수
    # 여름 6·7·8월이 기준범주
    # --------------------------------------------------------
    if "Month_Sold" in df.columns:
        month = pd.to_numeric(
            df["Month_Sold"],
            errors="coerce"
        )

        df["Spring"] = month.isin(
            [3, 4, 5]
        ).astype(int)

        df["Fall"] = month.isin(
            [9, 10, 11]
        ).astype(int)

        df["Winter"] = month.isin(
            [12, 1, 2]
        ).astype(int)

    return df


# ============================================================
# 3. XGBoost 실행 함수
# ============================================================
def run_xgboost_prediction(
    df_sub,
    feature_cols,
    target_col="Log_Price_per_m2",
    random_state=42
):
    """
    XGBoost 회귀모형

    - 데이터에 존재하는 설명변수만 사용
    - Train 70%, Test 30%
    - Train/Test R2 및 RMSE 계산
    - Test 예측값 저장
    - 변수 중요도 저장
    """

    df_model = df_sub.copy()

    # --------------------------------------------------------
    # 데이터에 실제로 존재하는 설명변수만 선택
    # 없는 설명변수는 조용히 제외
    # --------------------------------------------------------
    valid_features = [
        col for col in feature_cols
        if col in df_model.columns
    ]

    if target_col not in df_model.columns:
        raise ValueError(
            f"종속변수 '{target_col}'가 데이터에 없습니다."
        )

    if len(valid_features) == 0:
        raise ValueError(
            "사용할 수 있는 설명변수가 없습니다."
        )

    # --------------------------------------------------------
    # 사용 변수 숫자형 변환
    # --------------------------------------------------------
    for col in valid_features + [target_col]:
        df_model[col] = pd.to_numeric(
            df_model[col],
            errors="coerce"
        )

    # 무한대 값을 결측치로 처리
    df_model = df_model.replace(
        [np.inf, -np.inf],
        np.nan
    )

    # --------------------------------------------------------
    # 사용 변수와 종속변수의 결측치 제거
    # --------------------------------------------------------
    df_model = (
        df_model
        .dropna(
            subset=valid_features + [target_col]
        )
        .reset_index(drop=True)
    )

    if len(df_model) < 30:
        raise ValueError(
            f"결측치 제거 후 유효 표본이 부족합니다: "
            f"{len(df_model)}개"
        )

    X = df_model[valid_features]
    y = df_model[target_col]

    # --------------------------------------------------------
    # Train 70%, Test 30%
    # --------------------------------------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=random_state
    )

    # --------------------------------------------------------
    # XGBoost 모델
    # --------------------------------------------------------
    xgb = XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        objective="reg:squarederror",
        random_state=random_state,
        n_jobs=-1
    )

    # 모델 학습
    xgb.fit(X_train, y_train)

    # --------------------------------------------------------
    # Train 성능
    # --------------------------------------------------------
    y_pred_train = xgb.predict(X_train)

    r2_train = r2_score(
        y_train,
        y_pred_train
    )

    rmse_train = np.sqrt(
        mean_squared_error(
            y_train,
            y_pred_train
        )
    )

    # --------------------------------------------------------
    # Test 성능
    # --------------------------------------------------------
    y_pred_test = xgb.predict(X_test)

    r2_test = r2_score(
        y_test,
        y_pred_test
    )

    rmse_test = np.sqrt(
        mean_squared_error(
            y_test,
            y_pred_test
        )
    )

    # --------------------------------------------------------
    # Test 예측값
    # --------------------------------------------------------
    pred_df = pd.DataFrame({
        "Actual": y_test.to_numpy(),
        "Predicted": y_pred_test,
        "Residual": y_test.to_numpy() - y_pred_test
    }).reset_index(drop=True)

    # --------------------------------------------------------
    # 변수 중요도
    # --------------------------------------------------------
    importance_df = pd.DataFrame({
        "Feature": valid_features,
        "Importance": xgb.feature_importances_
    }).sort_values(
        by="Importance",
        ascending=False
    ).reset_index(drop=True)

    return {
        "model": xgb,
        "r2_train": r2_train,
        "rmse_train": rmse_train,
        "r2_test": r2_test,
        "rmse_test": rmse_test,
        "pred_df": pred_df,
        "importance_df": importance_df,
        "n_train": len(X_train),
        "n_test": len(X_test),
        "used_features": valid_features
    }


# ============================================================
# 4. 데이터 준비
# ============================================================
df_all = prepare_seoul_df_final(df_raw)


# ============================================================
# 5. 사용할 설명변수
# ============================================================
features = [
    "Size_Level",
    "Floor",
    "Units_Level",
    "Parking_per_Household",
    "Construction_Year",
    "Log_Dist_Subway",
    "Log_Dist_Green",
    "Log_Dist_Water",
    "Dist_CBD_km",
    "Sex_ratio_ratio",
    "Pop_Density_Level",
    "Medium_age_ratio",
    "Young_pop_ratio",
    "Old_pop_ratio",
    "Spring",
    "Fall",
    "Winter"
]

target = "Log_Price_per_m2"


# ============================================================
# 6. 연도별 XGBoost
# ============================================================
xgb_results_by_year = []
xgb_prediction_dict = {}
xgb_importance_dict = {}

for year in [2022, 2023]:

    if "Year_Sold" not in df_all.columns:
        raise ValueError(
            "데이터에 'Year_Sold' 변수가 없습니다."
        )

    year_values = pd.to_numeric(
        df_all["Year_Sold"],
        errors="coerce"
    )

    df_year = df_all[
        year_values == year
    ].copy()

    if len(df_year) < 30:
        print(
            f"{year}: 원자료 표본 수 부족으로 건너뜀"
        )
        continue

    try:
        xgb_out = run_xgboost_prediction(
            df_sub=df_year,
            feature_cols=features,
            target_col=target,
            random_state=42
        )

    except ValueError as error:
        print(f"{year}: {error}")
        continue

    xgb_results_by_year.append({
        "Year": year,
        "Train_R2": xgb_out["r2_train"],
        "Train_RMSE": xgb_out["rmse_train"],
        "Test_R2": xgb_out["r2_test"],
        "Test_RMSE": xgb_out["rmse_test"],
        "Train_N": xgb_out["n_train"],
        "Test_N": xgb_out["n_test"]
    })

    xgb_prediction_dict[year] = (
        xgb_out["pred_df"]
    )

    xgb_importance_dict[year] = (
        xgb_out["importance_df"]
    )

    print(f"\n[{year}년 XGBoost 결과]")

    print("\n사용 변수")
    print(xgb_out["used_features"])

    print("\nTrain 성능")
    print(
        f"R2   : {xgb_out['r2_train']:.4f}"
    )
    print(
        f"RMSE : {xgb_out['rmse_train']:.4f}"
    )

    print("\nTest 성능")
    print(
        f"R2   : {xgb_out['r2_test']:.4f}"
    )
    print(
        f"RMSE : {xgb_out['rmse_test']:.4f}"
    )

    print(
        f"\nTrain N : {xgb_out['n_train']:,}"
    )
    print(
        f"Test N  : {xgb_out['n_test']:,}"
    )

    print("\n변수 중요도 상위 10개")
    display(
        xgb_out["importance_df"].head(10)
    )


# ============================================================
# 7. 전체 기간 XGBoost
# ============================================================
xgb_full_out = run_xgboost_prediction(
    df_sub=df_all,
    feature_cols=features,
    target_col=target,
    random_state=42
)

print("\n[전체 기간 XGBoost 결과]")

print("\n사용 변수")
print(xgb_full_out["used_features"])

print("\nTrain 성능")
print(
    f"R2   : {xgb_full_out['r2_train']:.4f}"
)
print(
    f"RMSE : {xgb_full_out['rmse_train']:.4f}"
)

print("\nTest 성능")
print(
    f"R2   : {xgb_full_out['r2_test']:.4f}"
)
print(
    f"RMSE : {xgb_full_out['rmse_test']:.4f}"
)

print(
    f"\nTrain N : {xgb_full_out['n_train']:,}"
)
print(
    f"Test N  : {xgb_full_out['n_test']:,}"
)

print("\n전체 기간 변수 중요도 상위 10개")
display(
    xgb_full_out["importance_df"].head(10)
)


# ============================================================
# 8. 성능 비교표
# ============================================================
xgb_results_table = pd.DataFrame(
    xgb_results_by_year
)

full_row = pd.DataFrame([{
    "Year": "Full Sample",
    "Train_R2": xgb_full_out["r2_train"],
    "Train_RMSE": xgb_full_out["rmse_train"],
    "Test_R2": xgb_full_out["r2_test"],
    "Test_RMSE": xgb_full_out["rmse_test"],
    "Train_N": xgb_full_out["n_train"],
    "Test_N": xgb_full_out["n_test"]
}])

xgb_results_table = pd.concat(
    [xgb_results_table, full_row],
    ignore_index=True
)

print("\n[XGBoost 성능 비교표]")
display(xgb_results_table)


# ============================================================
# 9. 예측값 일부 확인
# ============================================================
for year, pred_df in xgb_prediction_dict.items():

    print(f"\n[{year}년 예측값 일부]")
    display(pred_df.head())

print("\n[전체 기간 예측값 일부]")
display(
    xgb_full_out["pred_df"].head()
)


# ============================================================
# 10. 엑셀 저장
# ============================================================
output_path = (
    "/content/drive/MyDrive/"
    "Seoul_Apartment_XGB_Results.xlsx"
)

with pd.ExcelWriter(
    output_path,
    engine="openpyxl"
) as writer:

    # 성능 비교표
    xgb_results_table.to_excel(
        writer,
        sheet_name="XGB_Performance",
        index=False
    )

    # 연도별 예측값과 변수 중요도
    for year in xgb_prediction_dict:

        xgb_prediction_dict[year].to_excel(
            writer,
            sheet_name=f"Pred_{year}",
            index=False
        )

        xgb_importance_dict[year].to_excel(
            writer,
            sheet_name=f"Importance_{year}",
            index=False
        )

    # 전체 기간 예측값
    xgb_full_out["pred_df"].to_excel(
        writer,
        sheet_name="Pred_Full",
        index=False
    )

    # 전체 기간 변수 중요도
    xgb_full_out["importance_df"].to_excel(
        writer,
        sheet_name="Importance_Full",
        index=False
    )

print(f"\n저장 완료: {output_path}")

Mounted at /content/drive

[2022년 XGBoost 결과]

사용 변수
['Size_Level', 'Floor', 'Units_Level', 'Parking_per_Household', 'Construction_Year', 'Log_Dist_Subway', 'Log_Dist_Green', 'Log_Dist_Water', 'Dist_CBD_km', 'Sex_ratio_ratio', 'Pop_Density_Level', 'Medium_age_ratio', 'Young_pop_ratio', 'Old_pop_ratio', 'Spring', 'Fall', 'Winter']

Train 성능
R2   : 0.9534
RMSE : 0.0971

Test 성능
R2   : 0.8849
RMSE : 0.1553

Train N : 7,405
Test N  : 3,174

변수 중요도 상위 10개


,Feature,Importance
0,Young_pop_ratio,0.219002
1,Units_Level,0.151458
2,Old_pop_ratio,0.098812
3,Construction_Year,0.094980
4,Sex_ratio_ratio,0.061723
5,Fall,0.052409
6,Pop_Density_Level,0.046137
7,Parking_per_Household,0.043486
8,Size_Level,0.037776
9,Spring,0.034109



[2023년 XGBoost 결과]

사용 변수
['Size_Level', 'Floor', 'Units_Level', 'Parking_per_Household', 'Construction_Year', 'Log_Dist_Subway', 'Log_Dist_Green', 'Log_Dist_Water', 'Dist_CBD_km', 'Sex_ratio_ratio', 'Pop_Density_Level', 'Medium_age_ratio', 'Young_pop_ratio', 'Old_pop_ratio', 'Spring', 'Fall', 'Winter']

Train 성능
R2   : 0.9526
RMSE : 0.0940

Test 성능
R2   : 0.9307
RMSE : 0.1141

Train N : 21,749
Test N  : 9,322

변수 중요도 상위 10개


,Feature,Importance
0,Young_pop_ratio,0.273908
1,Old_pop_ratio,0.131874
2,Units_Level,0.088680
3,Construction_Year,0.084261
4,Sex_ratio_ratio,0.061758
5,Log_Dist_Subway,0.055265
6,Parking_per_Household,0.053441
7,Pop_Density_Level,0.042545
8,Medium_age_ratio,0.038472
9,Size_Level,0.031487



[전체 기간 XGBoost 결과]

사용 변수
['Size_Level', 'Floor', 'Units_Level', 'Parking_per_Household', 'Construction_Year', 'Log_Dist_Subway', 'Log_Dist_Green', 'Log_Dist_Water', 'Dist_CBD_km', 'Sex_ratio_ratio', 'Pop_Density_Level', 'Medium_age_ratio', 'Young_pop_ratio', 'Old_pop_ratio', 'Spring', 'Fall', 'Winter']

Train 성능
R2   : 0.9204
RMSE : 0.1307

Test 성능
R2   : 0.9112
RMSE : 0.1371

Train N : 113,662
Test N  : 48,713

전체 기간 변수 중요도 상위 10개


...


[2023년 예측값 일부]


,Actual,Predicted,Residual
0,16.454920,16.413332,0.041588
1,16.102828,16.182549,-0.079720
2,16.659107,16.641830,0.017277
3,16.075036,16.153780,-0.078744
4,16.046305,15.999835,0.046470



[전체 기간 예측값 일부]


,Actual,Predicted,Residual
0,16.463642,16.490576,-0.026934
1,16.152554,16.215702,-0.063149
2,16.483932,16.093534,0.390398
3,16.355151,16.326988,0.028163
4,16.445963,16.519165,-0.073202



저장 완료: /content/drive/MyDrive/Seoul_Apartment_XGB_Results.xlsx
